# Creating GAT Journey Planner Agent Model

### Read in training data -> NetworkX graph

In [55]:
import pandas as pd
import networkx as nx

data_path = "C:\\Users\\sasab\\Documents\\Projects\\MaaS_AI\\Agents\\journey_planner\\data\\"
db_path = "C:\\Users\\sasab\\Documents\\Projects\\MaaS_AI\\Main_App\\data_bases\\"

#loading in csv file
route_edges_df = pd.read_csv(data_path+"pune_maas_journey_planner_data.csv") #pune_maas_journey_planner_data.csv - metro routing data table
timetable_l1 = pd.read_csv(db_path+"purple_line_timetable.csv") #timetable_data.csv - metro train timetable data
timetable_l2 = pd.read_csv(db_path+"aqua_line_timetable.csv")
timetable_l3 = pd.read_csv(db_path+"pink_line_timetable.csv")
swipes_df = pd.read_csv(db_path+"swipe_metadata.csv")  #swipe_metadata.csv - metro swipe data with each ticket pur

# clean columns data
route_edges_df.columns = route_edges_df.columns.str.strip()
timetable_l1.columns = timetable_l1.columns.str.strip()
timetable_l2.columns = timetable_l2.columns.str.strip()
timetable_l3.columns = timetable_l3.columns.str.strip()
swipes_df.columns = swipes_df.columns.str.strip()

# fix differnt name - normalize station
def clean_station_name(s):
    if pd.isna(s):
        return s
    return str(s).strip()

# Clean edge df station names
route_edges_df["station_name_from"] = route_edges_df["station_name_from"].apply(clean_station_name)
route_edges_df["station_name_to"] = route_edges_df["station_name_to"].apply(clean_station_name)

# Clean swipe df station names
swipes_df["Start station"] = swipes_df["Start station"].apply(clean_station_name)
swipes_df["End Station"] = swipes_df["End Station"].apply(clean_station_name)
swipes_df["Swipe In station"] = swipes_df["Swipe In station"].apply(clean_station_name)

# Map alternate names to one standard name
station_name_map = {
    "RamWadi": "Ramwadi",
    "Ruby Hall": "Ruby Hall Clinic",
    "Civil Court": "District Court (Civil Court)",
    "District Court Pune": "District Court (Civil Court)",
}

def apply_station_map(s):
    if pd.isna(s):
        return s
    return station_name_map.get(s, s)

for col in ["Start station", "End Station", "Swipe In station"]:
    swipes_df[col] = swipes_df[col].apply(apply_station_map)
    
# Parse swipe times 
swipes_df["Swipe in time"] = pd.to_datetime(
    swipes_df["Swipe in time"], format="%H:%M", errors="coerce"
)

swipes_df["Approximate boarding time"] = pd.to_datetime(
    swipes_df["Approximate boarding time"], format="%H:%M", errors="coerce"
)

swipes_df["swipe_hour"] = swipes_df["Swipe in time"].dt.hour
swipes_df["boarding_hour"] = swipes_df["Approximate boarding time"].dt.hour

#encode 'string' value attributes (line,mode)
#for every new value map to unique 
line_map = {name: i for i, name in enumerate(route_edges_df["line"].unique())}     #line: purple | pink | aqua
mode_map = {name: i for i, name in enumerate(route_edges_df["mode"].unique(), start=1)} # mode: metro | feeder bus

route_edges_df["line_id"] = route_edges_df["line"].map(line_map)
route_edges_df["mode_id"] = route_edges_df["mode"].map(mode_map)
route_edges_df["is_transfer"] = route_edges_df["is_transfer"].astype(int)
route_edges_df["bidirectional"] = route_edges_df["bidirectional"].astype(int)

#combine all timetable df
def reshape_timetable(tt_df, line_name):
    tt_long = tt_df.melt(
        id_vars=["Train ID", "Direction"],
        var_name="station_name",
        value_name="scheduled_time"
    )
    tt_long["station_name"] = tt_long["station_name"].apply(clean_station_name)
    tt_long["scheduled_time"] = pd.to_datetime(
        tt_long["scheduled_time"], format="%H:%M", errors="coerce"
    )
    tt_long["hour"] = tt_long["scheduled_time"].dt.hour
    tt_long["line_name"] = line_name
    return tt_long

purple_long = reshape_timetable(timetable_l1, "purple")
aqua_long = reshape_timetable(timetable_l2, "aqua")
pink_long = reshape_timetable(timetable_l3, "pink")

timetable_df = pd.concat([purple_long, aqua_long, pink_long], ignore_index=True)

#check
# print ("========route_edges table========")
# print (route_edges_df)

# print ("\n========timetable table========")
# print (timetable_df)

# print ("\n========swipes metadata table========")
# print (swipes_df)

### Create a station master list
| Feature            | Meaning      |
| ------------------ | ------------ |
| `transfer_station` | node feature |
| `is_transfer`      | edge feature |


In [50]:
stations_from = route_edges_df[["station_id_from", "station_name_from","is_transfer"]].rename(
    columns={"station_id_from": "station_id", "station_name_from": "station_name"}
)

stations_to = route_edges_df[["station_id_to", "station_name_to"]].rename(
    columns={"station_id_to": "station_id", "station_name_to": "station_name"}
)

stations_df = pd.concat([stations_from, stations_to], ignore_index=True).drop_duplicates()
stations_df = stations_df.sort_values("station_id").reset_index(drop=True)

#add transfer flag
transfer_flags = (
    route_edges_df.groupby("station_id_from")["is_transfer"]
    .max()
    .reset_index()
    .rename(columns={"station_id_from": "station_id", "is_transfer": "transfer_station"})
)

stations_df = stations_df.merge(transfer_flags, on="station_id", how="left")
stations_df["transfer_station"] = stations_df["transfer_station"].fillna(0).astype(int)

# print(stations_df)

### Building Feature


In [ ]:
#count train stops at station per hr
trains_stopping = (
    timetable_df.groupby(["station_name", "hour"])
    .size()
    .reset_index(name="trains_stopping")
)

#define  swipe based features
# Boardings = people swiping into a station during an hour
boardings = (
    swipes_df.groupby(["Swipe In station", "swipe_hour"])
    .size()
    .reset_index(name="expected_boardings")
    .rename(columns={"Swipe In station": "station_name", "swipe_hour": "hour"})
)

# Alightings = people whose end station is that station during an hour
alightings = (
    swipes_df.groupby(["End Station", "boarding_hour"])
    .size()
    .reset_index(name="expected_alightings")
    .rename(columns={"End Station": "station_name", "boarding_hour": "hour"})
)

# Feeder requests = people requesting feeder service for that end station during an hour
feeder_requests = (
    swipes_df[
        swipes_df["Feeder Service (Y/N)"]
        .astype(str)
        .str.strip()
        .str.upper()
        .isin(["Y", "YES", "TRUE", "1"])
    ]
    .groupby(["End Station", "boarding_hour"])
    .size()
    .reset_index(name="feeder_requests")
    .rename(columns={"End Station": "station_name", "boarding_hour": "hour"})
)

#check
# print("=============train stop=============")
# print (trains_stopping)

# print("\n=============boardings=============")
# print (boardings)

# print("\n=============alightings=============")
# print (alightings)

# print("\n=============feeder requests=============")
# print (feeder_requests)


### CREATE STATION-HOUR GRID

In [58]:
#every station should have a row for every hour in the timetable range
hours_df = pd.DataFrame({
    "hour": sorted(timetable_df["hour"].dropna().unique())
})

station_hour_df = (
    stations_df.assign(key=1)
    .merge(hours_df.assign(key=1), on="key")
    .drop("key", axis=1)
)

print("\n=============Hours=============")
print (hours_df)

print("\n=============station_hour=============")
print (station_hour_df)


=============Hours=============
   hour
0     6
1     7
2     8
3     9
4    10
5    11
6    12
7    13
8    14
9    15

=============station_hour=============
     station_id   station_name  is_transfer  transfer_station  hour
0           A01  Chandni Chowk          0.0                 0     6
1           A01  Chandni Chowk          0.0                 0     7
2           A01  Chandni Chowk          0.0                 0     8
3           A01  Chandni Chowk          0.0                 0     9
4           A01  Chandni Chowk          0.0                 0    10
...         ...            ...          ...               ...   ...
1465        P23         Katraj          NaN                 0    11
1466        P23         Katraj          NaN                 0    12
1467        P23         Katraj          NaN                 0    13
1468        P23         Katraj          NaN                 0    14
1469        P23         Katraj          NaN                 0    15

[1470 rows x 5 columns

In [59]:
#every station should have a row for every hour in the timetable range
hours_df = pd.DataFrame({
    "hour": sorted(timetable_df["hour"].dropna().unique())
})

station_hour_df = (
    stations_df.assign(key=1)
    .merge(hours_df.assign(key=1), on="key")
    .drop("key", axis=1)
)

### Build Feat Table

In [60]:
#MERGE FEATURES INTO ONE TABLE
station_features_df = station_hour_df.merge(
    trains_stopping, on=["station_name", "hour"], how="left"
)

station_features_df = station_features_df.merge(
    boardings, on=["station_name", "hour"], how="left"
)

station_features_df = station_features_df.merge(
    alightings, on=["station_name", "hour"], how="left"
)

station_features_df = station_features_df.merge(
    feeder_requests, on=["station_name", "hour"], how="left"
)

for col in ["trains_stopping", "expected_boardings", "expected_alightings", "feeder_requests"]:
    station_features_df[col] = station_features_df[col].fillna(0).astype(int)

### CREATE SIMPLE LABELS

In [61]:
# Simple congestion score for V1
station_features_df["congestion_score"] = (
    station_features_df["expected_boardings"]
    + station_features_df["expected_alightings"]
    + station_features_df["feeder_requests"]
    + station_features_df["transfer_station"]
)

def label_congestion(score):
    if score < 3:
        return 0   # low
    elif score < 8:
        return 1   # medium
    return 2       # high

station_features_df["congestion_label"] = station_features_df["congestion_score"].apply(label_congestion)


# Feeder label for V1
def label_feeder(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    elif n == 2:
        return 2
    return 3   # 3+

station_features_df["feeder_label"] = station_features_df["feeder_requests"].apply(label_feeder)



###  SAVE OUTPUT

In [62]:
# Keep only the simpler V1 columns
final_cols = [
    "station_id",
    "station_name",
    "hour",
    "transfer_station",
    "trains_stopping",
    "expected_boardings",
    "expected_alightings",
    "feeder_requests",
    "congestion_label",
    "feeder_label",
]

station_features_df = station_features_df[final_cols]

station_features_df.to_csv("station_features.csv", index=False)

print("Saved station_features.csv")
print(station_features_df.head(20))
print(station_features_df.shape)

Saved station_features.csv
   station_id       station_name  hour  transfer_station  trains_stopping  \
0         A01      Chandni Chowk     6                 0               10   
1         A01      Chandni Chowk     7                 0               10   
2         A01      Chandni Chowk     8                 0               10   
3         A01      Chandni Chowk     9                 0               10   
4         A01      Chandni Chowk    10                 0               10   
5         A01      Chandni Chowk    11                 0               10   
6         A01      Chandni Chowk    12                 0               10   
7         A01      Chandni Chowk    13                 0               10   
8         A01      Chandni Chowk    14                 0                1   
9         A01      Chandni Chowk    15                 0                0   
10        A02  Kothrud Bus Depot     6                 0               10   
11        A02  Kothrud Bus Depot     7           

### Define NetworkX graph node/edge structure

In [ ]:
# define NetworkX graph
G = nx.DiGraph()

# Add node features
# mode_nodes = set()  #transport mode  
# transfer_nodes = set()  #transport mode 

#loop through rows and set edges
for _, row in route_edges_df.iterrows():
    G.add_node(
        row["station_id_from"],
        station_name = row["station_name_from"],
        mode_id=row["mode_id"],
        transfer=int(row["is_transfer"]),
        line_id=row["line_id"]
    )
    G.add_node(
        row["station_id_to"],
        station_name = row["station_name_to"],
        mode_id=row["mode_id"],
        transfer=int(row["is_transfer"]),
        line_id=row["line_id"]
    )
    
    G.add_edge(
        row["station_id_from"],
        row["station_id_to"],
        line_id=row["line_id"],
        length=float(row["distance_km"]),
        minutes=float(row["travel_time_min"]),
        transfer=int(row["is_transfer"]),
        transfer_time=float(row["transfer_penalty_min"])
    )

    if row["bidirectional"]:
        G.add_edge(
            row["station_id_to"],
            row["station_id_from"],
            line_id=row["line_id"],
            length=float(row["distance_km"]),
            minutes=float(row["travel_time_min"]),
            transfer=int(row["is_transfer"]),
            transfer_time=float(row["transfer_penalty_min"])
        )        
print(f"Nodes: {list(G.nodes)}")
print(f"Edges: {list(G.edges)}")


Nodes: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'K01', 'K02', 'K03', 'K04', 'K05', 'K06', 'K07', 'K08', 'K09', 'K10', 'K11', 'K12', 'K13', 'K14', 'K15', 'K16', 'K17', 'K18', 'K19', 'K20', 'K21', 'K22', 'K23']
Edges: [('P01', 'P02'), ('P02', 'P01'), ('P02', 'P03'), ('P03', 'P02'), ('P03', 'P04'), ('P04', 'P03'), ('P04', 'P05'), ('P05', 'P04'), ('P05', 'P06'), ('P06', 'P05'), ('P06', 'P07'), ('P07', 'P06'), ('P07', 'P08'), ('P08', 'P07'), ('P08', 'P09'), ('P09', 'P08'), ('P09', 'P10'), ('P10', 'P09'), ('P10', 'P11'), ('P11', 'P10'), ('P11', 'P12'), ('P12', 'P11'), ('P12', 'P13'), ('P13', 'P12'), ('P13', 'P14'), ('P14', 'P13'), ('P14', 'P15'), ('P14', 'K22'), ('P15', 'P1

### Convert Network X graph to PyG dataset to train model

In [5]:
import torch
from torch_geometric.utils.convert import from_networkx
import torch.nn.functional as F
import torch_geometric.transforms as T

for node in G.nodes():
    G.nodes[node]["x"] = G.nodes[node]["mode_id"] 
    
# Convert to PyG
pyg_data = from_networkx(
    G,
    group_node_attrs=["x"],
    group_edge_attrs=["line_id", "length", "minutes", "transfer", "transfer_time"]
)

print(pyg_data)
print("Edge index shape:", pyg_data.edge_index.shape)
print("Edge attr shape:", pyg_data.edge_attr.shape)
print("num of node features:", pyg_data.num_node_features)

Data(edge_index=[2, 152], station_name=[75], mode_id=[75], transfer=[75], line_id=[75], x=[75, 1], edge_attr=[152, 5])
Edge index shape: torch.Size([2, 152])
Edge attr shape: torch.Size([152, 5])
num of node features: 1


In [ ]:
from torch_geometric.nn import GATv2Conv

class GAT(torch.nn.Module):
    # in_channels - number of node feature
    # hidden_channels - 16
    # out_channels - number of classes output channels.set to 3, will rep congestion level to use in Dispact Agent layer
    # edge_features - number of edge feature
    def __init__(self, in_channels, hidden_channels, out_channels,edge_features):
        super().__init__()
        # First GAT layer with multi-head attention
        # self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.6)
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=8, edge_dim=edge_features)
        # Second GAT layer (output layer)
        # self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)
        self.conv2 = GATv2Conv(hidden_channels * 8,out_channels,heads=1,concat=False,edge_dim=edge_features)
        
    #dropout - default .6 or 60% once outr dataset is small and doent have alot of node feature
        #new default set to 0.1-0.2 for minimal regularization and small graphs
    def forward(self, x, edge_index, edge_attr):
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.elu(self.conv1(x, edge_index,edge_attr))
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(x, edge_index,edge_attr)
        return F.log_softmax(x, dim=1)

#create GAT model
num_classes =  3
edge_features = pyg_data.edge_attr.shape[1]
model = GAT(pyg_data.num_node_features, 16,num_classes,edge_features)

# Sanity checks
def test_check():
    print("=== Input Graph Shapes ===") #orginal shape
    print("x shape:", pyg_data.x.shape)
    print("edge_index shape:", pyg_data.edge_index.shape)
    print("edge_attr shape:", pyg_data.edge_attr.shape)

    model.eval()
    out = model(pyg_data,pyg_data.edge_attr.shape) #gat model shape



torch.Size([152, 5])


TypeError: dropout(): argument 'input' (position 1) must be Tensor, not Data

### creating Trainning/Val/Test masks

Total nodes = 75

| Split      | Percentage | Nodes        |
| ---------- | ---------- | ------------ |
| Training   | **60%**    | **45 nodes** |
| Validation | **20%**    | **15 nodes** |
| Test       | **20%**    | **15 nodes** |


In [35]:
num_nodes = pyg_data.num_nodes

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

rand_perm = torch.randperm(num_nodes)

train_mask[rand_perm[:45]] = True
val_mask[rand_perm[45:60]] = True
test_mask[rand_perm[60:]] = True

pyg_data.train_mask = train_mask
pyg_data.val_mask = val_mask
pyg_data.test_mask = test_mask

print(f"num_train_nodes: {sum(train_mask)} | num_val_nodes: {sum(val_mask)} | num_test_nodes: {sum(test_mask)}")

num_train_nodes: 45 | num_val_nodes: 15 | num_test_nodes: 15


### Create GAT base model before trainning

In [31]:
from torch_geometric.nn import GATv2Conv

class GAT(torch.nn.Module):
    # in_channels - number of node feature
    # hidden_channels - 16
    # out_channels - number of classes output channels.set to 3, will rep congestion level to use in Dispact Agent layer
    # edge_features - number of edge feature
    def __init__(self, in_channels, hidden_channels, out_channels,edge_features):
        super().__init__()
        # First GAT layer with multi-head attention
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=8, edge_dim=edge_features)
        # Second GAT layer (output layer)
        self.conv2 = GATv2Conv(hidden_channels * 8,out_channels,heads=1,concat=False,edge_dim=edge_features)
        
    #dropout - default .6 or 60% once outr dataset is small and doent have alot of node feature
        #new default set to 0.1-0.2 for minimal regularization and small graphs
    def forward(self, x, edge_index, edge_attr):
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.elu(self.conv1(x, edge_index,edge_attr))
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(x, edge_index,edge_attr)
        return F.log_softmax(x, dim=1)

#create GAT model
num_classes =  3
edge_features = pyg_data.edge_attr.shape[1]
gat_model = GAT(pyg_data.num_node_features, 16,num_classes,edge_features)
optimizer = torch.optim.Adam(gat_model.parameters(), lr=0.005, weight_decay=5e-4)

In [32]:
print("=== Orginal datatypes ===")
print("x dtype:", pyg_data.x.dtype)
print("edge_index dtype:", pyg_data.edge_index.dtype)
print("edge_attr dtype:", pyg_data.edge_attr.dtype)

#convert to correct datatypes
pyg_data.x = pyg_data.x.float()
pyg_data.edge_attr = pyg_data.edge_attr.float()
pyg_data.edge_index = pyg_data.edge_index.long()

print("\n=== Corrected datatypes ===")
print("x dtype:", pyg_data.x.dtype)
print("edge_index dtype:", pyg_data.edge_index.dtype)
print("edge_attr dtype:", pyg_data.edge_attr.dtype)

=== Orginal datatypes ===
x dtype: torch.float32
edge_index dtype: torch.int64
edge_attr dtype: torch.float32

=== Corrected datatypes ===
x dtype: torch.float32
edge_index dtype: torch.int64
edge_attr dtype: torch.float32


In [33]:
# Sanity checks
def test_check():
    print("=== Input Graph Shapes ===") #orginal shape
    print("x shape:", pyg_data.x.shape)
    print("edge_index shape:", pyg_data.edge_index.shape)
    print("edge_attr shape:", pyg_data.edge_attr.shape)

    gat_model.eval()
    with torch.no_grad():
        out = gat_model(pyg_data.x,pyg_data.edge_index,pyg_data.edge_attr) #gat model shape without gradient change
        print("\n=== Model Output Shape ===")
        print("out shape:", out.shape)
    
test_check()


=== Input Graph Shapes ===
x shape: torch.Size([75, 1])
edge_index shape: torch.Size([2, 152])
edge_attr shape: torch.Size([152, 5])

=== Model Output Shape ===
out shape: torch.Size([75, 3])


### Create training/test eval functions

In [ ]:
def train():
    gat_model.train()
    optimizer.zero_grad()
    # Forward pass
    out = gat_model(pyg_data.x,pyg_data.edge_index,pyg_data.edge_attr)
    
    # Calculate loss on training nodes only
    loss = F.nll_loss(out[train_mask], pyg_data.y[train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

#test function to test the trainning accuracy
def test(model):
    model.eval()
    with torch.no_grad():
        #preiction on the whole graph
        logits = model(pyg_data.x,pyg_data.edge_index,pyg_data.edge_attr)
        
        #only focus on training_mask
        loss = F.nll_loss(logits[pyg_data.train_mask], pyg_data.y[pyg_data.train_mask])
        loss.backward()
        
        #compute accuracy
        preds = logits.argmax(dim=-1)
        acc = (preds[pyg_data.train_mask]==pyg_data.y[pyg_data.train_mask]).float().mean()
        
        return acc
    
# funct to vaildate model
def evaluate(model):
    model.eval()
    with torch.no_grad():
        
        #forward
        logits = model(pyg_data.x,pyg_data.edge_index,pyg_data.edge_attr)
        loss = F.nll_loss(logits[pyg_data.train_mask], pyg_data.y[pyg_data.train_mask])
        
        #compute accuracy
        preds = logits.argmax(dim=-1) #pred = logits[mask].max(1)[1]
        acc = (preds[pyg_data.val_mask]==pyg_data.y[pyg_data.val_mask]).float().mean()
        
        return loss, acc     

epoch_num = 40

# Training loop over epochs
for epoch in range(epoch_num):
    train_loss = train()
    train_acc = test(gat_model)
    val_loss, val_acc = evaluate(gat_model)
    
    if epoch%5==0 or epoch == epoch_num-1:
        print(f"Epoch: {epoch} | train_loss: {train_loss:.4f} | train_acc: {train_acc*100:.2f}% |" f" val_loss: {val_loss:.4f} | val_acc: {val_acc*100:.2f}%")
        


TypeError: 'NoneType' object is not subscriptable

# Feature
| Name               | Feature 1  |    Feature 2     | Feature 3   | Feature 4  |     Feature 5  |
| ------------------ | ---------- | ---------------- | ---------   | ---------- | ---------------|
| Node               | station_id | transfer_station | boardings   | alightings | feeder_requests|
| Edge               | from       | to               | travel_time | line       | is_transfer    |
| Label              | station    | congestion_class | feeder_class                              |
